# GRACE H1 — Full pipeline

Cache → train head → common top50 → semantic top5 → lexical/AST → CPG/prompt → Vertex → evaluation paired. Mặc định full test Devign; thêm reveal trong config để chạy ReVeal.

Dùng kernel WSL hiện có nếu cần sinh graph thiếu. GPU chỉ dành cho head nhỏ, không encode CodeT5 lại. Cell inference gọi API khi RUN_INFERENCE=True.

In [ ]:
from pathlib import Path
import os,sys,json,subprocess,importlib.util
candidates=([Path(os.environ['GRACE_H1_ROOT'])] if os.environ.get('GRACE_H1_ROOT') else [])
candidates += [Path.cwd(),*Path.cwd().parents,
               Path('D:/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection'),
               Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection')]
ROOT=next((p.resolve() for p in candidates if (p/'grace_h1').is_dir()),None)
if ROOT is None: raise RuntimeError('Set GRACE_H1_ROOT to this project folder')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
INSTALL_DEPENDENCIES=True
RUN_INFERENCE=True
TRAINED_BUNDLE=None
EXPORT_KAGGLE_INPUTS=False
config=json.loads((ROOT/'configs'/'experiment.json').read_text(encoding='utf-8'))
# Optional: config['datasets']=['devign','reveal']
# Optional CE LLM control: config['llm_arms'].append('h1_ce')
print('Python:',sys.executable,'Project:',ROOT)
print(json.dumps(config,indent=2))

In [ ]:
if INSTALL_DEPENDENCIES:
    if importlib.util.find_spec('torch') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','torch>=2.6,<3',
                               '--index-url','https://download.pytorch.org/whl/cpu'])
    subprocess.check_call([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements.txt')])

## 1. Tài nguyên đã copy

Dataset/embedding được kiểm tra ID, thứ tự và checksum tại stage sử dụng. Graph features thiếu được Joern tạo ở bước prepare.

In [ ]:
from grace_h1.io import read_json,write_json,load_local_environment
from grace_h1.experiment import train_all,prepare_all,preflight_all,infer_all,evaluate_all
from grace_h1.portable import create_bundle,import_bundle
import pandas as pd
load_local_environment(ROOT)
assets=read_json(ROOT/'data'/'manifest.json')
display(pd.DataFrame([{'dataset':d,'split':s,'rows':m['count']} for d,splits in assets['datasets'].items() for s,m in splits.items()]))
summary_path=ROOT/'provenance'/'cache_import.json'
if summary_path.exists(): print(json.dumps(read_json(summary_path),indent=2))
if TRAINED_BUNDLE: print(import_bundle(ROOT,Path(TRAINED_BUNDLE),'trained_heads'))
if EXPORT_KAGGLE_INPUTS:
    print(create_bundle(ROOT,ROOT/'bundles'/'grace-h1-training-inputs.zip','training_inputs'))

## 2. Học projection

CE control và CE + SupCon + hard negatives. Gradient/normalization dùng train-only; validation chọn checkpoint theo MCC của head. Đây là proxy, chưa chứng minh GRACE tăng điểm. Chạy lại resume/reuse checkpoint khớp. Có thể train trên Kaggle và import bundle trước cell này.

In [ ]:
selection=train_all(ROOT,config)
head_rows=[]
for dataset,heads in selection['heads'].items():
    for objective,h in heads.items():
        head_rows.append({'dataset':dataset,'objective':objective,'parameters':h['parameters'],
                          'device':h['device'],'validation_mcc':h['best']['metrics']['mcc'],
                          'validation_f1':h['best']['metrics']['f1'],'best_epoch':h['best']['epoch'],
                          'peak_gpu_MB':(h.get('peak_gpu_allocated_bytes') or 0)/1024**2})
display(pd.DataFrame(head_rows))

## 3. Chuẩn bị full cohort và đóng băng prompt

Baseline whitening 256 chiều fit trên train. H1 projection 128 chiều. Các arm giữ cùng top50, reranking, graph, template và LLM. Đặt JOERN_HOME trong .env hoặc config nếu cần sinh graph thiếu; không giảm train/test cohort.

In [ ]:
run_rows=prepare_all(ROOT,config)
display(pd.DataFrame(run_rows))

## 4. Kế hoạch request

Tổng số còn thiếu theo từng arm là cận trên: prompt trùng giữa các arm sẽ được reuse khi chạy tuần tự. Requests/token thực nằm trong inference summary.

In [ ]:
plans=preflight_all(ROOT,config)
display(pd.DataFrame([{k:p[k] for k in ['dataset','split','arm','queries','cached_queries','unique_prompts','new_unique_requests','worst_case_attempts']} for p in plans]))

## 5. Inference Vertex và resume

Cần ADC đang dùng trong WSL, VERTEX_PROJECT và VERTEX_LOCATION. Model/temperature/seed/max output lấy từ configs/reference_contract.yaml. Cache chỉ reuse khi exact prompt/model/decoding khớp. Request lỗi sau retry được lưu và chạy lại ở lượt sau.

In [ ]:
inference_results=infer_all(ROOT,config,allow_api=RUN_INFERENCE)
display(pd.DataFrame([{k:r.get(k) for k in ['dataset','split','arm','status','cached_queries','new_unique_requests','api_attempts_this_run','request_error_queries','wall_seconds']} for r in inference_results]))

## 6. Evaluation và paired comparison

F1/MCC tính trên prediction valid; invalid rate và recall trên mọi positive được báo riêng. Bootstrap dùng cùng tập query valid ở hai arm; không ép lỗi parse thành nhãn.

In [ ]:
report=evaluate_all(ROOT,config)
rows=[]
for r in report['scores']:
    m=r['metrics'];i=r['inference']
    rows.append({'dataset':r['dataset'],'split':r['split'],'arm':r['arm'],'n':m['n'],
                 'f1_valid':m['f1'],'mcc_valid':m['mcc'],'recall_all_positives':m['recall_all_positives'],
                 'invalid_rate':m['invalid_rate'],'new_requests':i['new_unique_requests'],
                 'api_attempts':i['api_attempts_this_run'],'new_usage':i['provider_usage_new_requests']})
display(pd.DataFrame(rows))
print('Status:',report['status'])
for comparison in report['comparisons']: print(json.dumps(comparison,indent=2))
print('Saved:',ROOT/'runs'/'experiment_summary.json')

## 7. Xuất head để chuyển máy

Bundle chỉ gồm artifact của head; không có API keys. Đổi seed/config trước khi train nếu chạy thêm ablation.

In [ ]:
EXPORT_TRAINED_HEADS=False
if EXPORT_TRAINED_HEADS:
    print(create_bundle(ROOT,ROOT/'bundles'/'grace-h1-trained-heads.zip','trained_heads'))